In [ ]:
#### Add nosie Denose vs real ####
import os
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd
import random

df = pd.read_csv("/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/add_noise_denoise/random_add_noise_step/train_real_fake_gap.csv")
base_image_dir = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/add_noise_denoise/random_add_noise_step"

num_samples = 4
random_indices = random.sample(range(len(df)), min(num_samples, len(df)))
rows = [df.iloc[idx] for idx in random_indices]

plt.figure(figsize=(16, 8), dpi=300)
plt.suptitle("Add noise Denoise vs Real - 4 Samples Comparison", fontsize=14, fontweight='bold', y=0.98)

for i, row in enumerate(rows):
    image_id = row["uid"]
    prompt = row["prompt"]
    
    fake_image_path = os.path.join(base_image_dir, "fake", f"{image_id}.png")
    real_image_path = os.path.join(base_image_dir, "real", f"{image_id}.png")
    
    fake_image = Image.open(fake_image_path)
    real_image = Image.open(real_image_path)
    
    row_idx = i // 2
    col_start = (i % 2) * 2  # 每个 image_id 占 2 列
    
    # Fake image
    plt.subplot(2, 4, row_idx * 4 + col_start + 1)
    plt.imshow(fake_image)
    plt.title(f"Fake\n{image_id}", fontsize=10)
    plt.axis("off")
    
    # Real image
    plt.subplot(2, 4, row_idx * 4 + col_start + 2)
    plt.imshow(real_image)
    plt.title(f"Real\n{image_id}", fontsize=10)
    plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
#### high_quality_train-crop 512 ####
import os
from PIL import Image
from tqdm import tqdm

source_image_dir = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/paired_real_generated_dataset/high_quality_train/real"
target_image_dir = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/paired_real_generated_dataset/high_quality_train/real-center_crop_512"

os.makedirs(target_image_dir, exist_ok=True)

def center_crop(image, crop_size=512):
    width, height = image.size
    left = (width - crop_size) // 2
    top = (height - crop_size) // 2
    right = left + crop_size
    bottom = top + crop_size
    return image.crop((left, top, right, bottom))

valid_exts = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp')

for filename in tqdm(os.listdir(source_image_dir)[0:1000], desc="Cropping images"):
    if not filename.lower().endswith(valid_exts):
        continue

    src_path = os.path.join(source_image_dir, filename)
    dst_name = os.path.splitext(filename)[0] + ".png"
    dst_path = os.path.join(target_image_dir, dst_name)

    try:
        with Image.open(src_path) as img:
            img = img.convert("RGB")
            cropped = center_crop(img, 512)
            cropped.save(dst_path, format="PNG")
    except Exception as e:
        print(f"❌ Error processing {filename}: {e}")

In [ ]:
import os
from PIL import Image
from tqdm import tqdm

source_image_dir = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/paired_real_generated_dataset/high_quality_train/real"
target_image_dir = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/paired_real_generated_dataset/high_quality_train/real-resize_1024-center_crop_512"

os.makedirs(target_image_dir, exist_ok=True)

def center_crop(image, crop_size=512):
    width, height = image.size
    left = (width - crop_size) // 2
    top = (height - crop_size) // 2
    right = left + crop_size
    bottom = top + crop_size
    return image.crop((left, top, right, bottom))

def resize_image(image, target_size=1024):
    width, height = image.size
    if width > height:
        new_width = target_size
        new_height = int((target_size / width) * height)
    else:
        new_height = target_size
        new_width = int((target_size / height) * width)
    
    return image.resize((new_width, new_height), Image.Resampling.LANCZOS)

valid_exts = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp')

for filename in tqdm(os.listdir(source_image_dir)[0:1000], desc="Processing images"):
    if not filename.lower().endswith(valid_exts):
        continue

    src_path = os.path.join(source_image_dir, filename)
    dst_name = os.path.splitext(filename)[0] + ".png"
    dst_path = os.path.join(target_image_dir, dst_name)

    try:
        with Image.open(src_path) as img:
            img = img.convert("RGB")
            width, height = img.size
            min_side = min(width, height)

            if min_side < 512:
                continue  # 忽略最小边小于 512 的图像
            elif min_side >= 512 and min_side < 1024:
                cropped = center_crop(img, 512)  # 最小边在 512 到 1024 之间，直接裁剪
            else:
                img_resized = resize_image(img, 1024)  # 最小边大于 1024，先缩放
                cropped = center_crop(img_resized, 512)  # 然后进行中心裁剪 512

            cropped.save(dst_path, format="PNG")
    except Exception as e:
        print(f"❌ Error processing {filename}: {e}")

print("✅ 所有图像已完成处理。")


In [ ]:
#### Fetch prompt from all-train-demo.csv ####
import csv
import os

csv_path = "/data_center/data2/dataset/chenwy/21164-data/all-train-demo.csv"
output_dir = "/data3/chenweiyan/notebook/fine-tune-diffusion/spo_gitee/DiffusionNFT/dataset/x_aigd"
output_file = os.path.join(output_dir, "test.txt")

os.makedirs(output_dir, exist_ok=True)

prompts = []
with open(csv_path, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        prompt = row.get('PROMPT', '').strip()
        text_length_str = row.get('Text Length', '').strip()

        if not text_length_str.isdigit():
            continue

        text_length = int(text_length_str)

        if prompt and text_length <= 40:
            prompts.append(prompt)

        if len(prompts) >= 1000:
            break

with open(output_file, 'w', encoding='utf-8') as f:
    f.write('\n'.join(prompts))

print(f"Successfully extracted and wrote {len(prompts)} prompts (Text Length <= 40) to {output_file}")
